In [1]:
import os
import json
from pathlib import Path

In [14]:
ENRON_DIR = "../../data/enronmail_raw"     # 主目录
OUTPUT_FILE = "mails.jsonl"

In [7]:
def parse_email(path):
    """
    将 enron 式邮件文本解析为 dict，包括 From, To, Subject, Date, Body
    """
    with open(path, "r", encoding="latin-1") as f:
        lines = f.readlines()

    headers = {}
    body_lines = []
    in_headers = True

    for line in lines:
        if in_headers:
            if line.strip() == "":
                in_headers = False
                continue
            # 解析 header 行
            if ":" in line:
                key, val = line.split(":", 1)
                headers[key.strip().lower()] = val.strip()
        else:
            body_lines.append(line.rstrip())

    return {
        "path": str(path),
        "from": headers.get("from", ""),
        "to": headers.get("to", ""),
        "subject": headers.get("subject", ""),
        "date": headers.get("date", ""),
        "ID": headers.get("message-id", ""),
        "body": "\n".join(body_lines),
        "rag_txt": "".join(lines[1:])
    }

def get_total_email(path):
    with open(path, "r", encoding="latin-1") as f:
        lines = f.readlines()

    return "".join(lines)

In [8]:
def all_emails(root_dir):
    """
    遍历所有文件夹并获得 email 文件路径
    """
    all_files = []
    for root, dirs, files in os.walk(root_dir):
        # print(f"Scanning {root} ...")
        for f in files:
            # Enron 数据基本都是 txt 或无后缀
            if f.lower().endswith(".txt") or f.lower().endswith(".") or "." not in f:
                all_files.append(Path(root) / f)
    return all_files

In [9]:
all_files_paths = all_emails(ENRON_DIR)
print(f"找到 {len(all_files_paths)} 封邮件样本，示例路径：{all_files_paths[:3]}")

找到 517401 封邮件样本，示例路径：[PosixPath('../../data/enronmail_raw/keavey-p/sent_items/5.'), PosixPath('../../data/enronmail_raw/keavey-p/sent_items/14.'), PosixPath('../../data/enronmail_raw/keavey-p/sent_items/36.')]


In [10]:
docs=[]

for email_path in all_files_paths:
    doc = parse_email(email_path)
    docs.append(doc)

In [15]:
docs[0]

{'path': '../../data/enronmail_raw/keavey-p/sent_items/5.',
 'from': 'peter.keavey@enron.com',
 'to': 'annemoisan@hotmail.com',
 'subject': '26JUL HOUSTON TO NEW YORK = ANNE MOISAN = TICKETED',
 'date': 'Mon, 14 May 2001 10:47:00 -0700 (PDT)',
 'ID': '<15792193.1075845174530.JavaMail.evans@thyme>',
 'body': '\n---------------------- Forwarded by Peter F Keavey/HOU/ECT on 05/14/2001 07:47 AM ---------------------------\n\n\nsandra delgado <sdelgado_vitoltvl@yahoo.com> on 05/11/2001 04:32:50 PM\nTo:\tPETER.F.KEAVEY@ENRON.COM\ncc:\nSubject:\t26JUL HOUSTON TO NEW YORK = ANNE MOISAN = TICKETED\n\n\n                                          AGENT JH/SS BOOKING REF\nZDJD9T\n\n                                          MOISAN/ANNE\n\n\n  ENRON\n  1400 SMITH\n  HOUSTON TX 77002\n  PETE KEAVEY X37277\n\n\n  DATE:  MAY 11 2001                   ENRON\n\nSERVICE               DATE  FROM           TO             DEPART\nARRIVE\n\nCONTINENTAL AIRLINES  26JUL HOUSTON TX     NEW YORK NY    1058A   328P

In [16]:
rag_texts_output = [
    {
        "_id": item["ID"],
        "title": f"{item['subject']}",
        "text": item["rag_txt"],
        "metadata":{
            "path": item["path"],
            "from": item["from"],
            "to": item["to"],
            "date": item["date"]
        }
    }
    for item in docs]

In [17]:
rag_texts_output[0]

{'_id': '<15792193.1075845174530.JavaMail.evans@thyme>',
 'title': '26JUL HOUSTON TO NEW YORK = ANNE MOISAN = TICKETED',
 'text': 'Date: Mon, 14 May 2001 10:47:00 -0700 (PDT)\nFrom: peter.keavey@enron.com\nTo: annemoisan@hotmail.com\nSubject: 26JUL HOUSTON TO NEW YORK = ANNE MOISAN = TICKETED\nCc: pkeavey@hotmail.com\nMime-Version: 1.0\nContent-Type: text/plain; charset=us-ascii\nContent-Transfer-Encoding: 7bit\nBcc: pkeavey@hotmail.com\nX-From: Peter F Keavey\nX-To: annemoisan <annemoisan@hotmail.com>\nX-cc: pkeavey <pkeavey@hotmail.com>\nX-bcc: \nX-Folder: \\Keavey, Peter F.\\Keavey, Peter F.\\Sent Items\nX-Origin: KEAVEY-P\nX-FileName: Keavey, Peter F..pst\n\n\n---------------------- Forwarded by Peter F Keavey/HOU/ECT on 05/14/2001 07:47 AM ---------------------------\n\n\nsandra delgado <sdelgado_vitoltvl@yahoo.com> on 05/11/2001 04:32:50 PM\nTo:\tPETER.F.KEAVEY@ENRON.COM\ncc:\t \nSubject:\t26JUL HOUSTON TO NEW YORK = ANNE MOISAN = TICKETED\n\n\n                                   

In [18]:
with open(OUTPUT_FILE, "w", encoding="utf-8") as out:
    for item in rag_texts_output:
        out.write(json.dumps(item, ensure_ascii=False) + "\n")